In [1]:
# =====================================
# STEP 1: Install dependencies
# =====================================
!pip install -q transformers torch accelerate bitsandbytes sentence-transformers faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.1 MB/s eta 0:00:00


In [2]:
# =====================================
# STEP 2: Load MedAlpaca with 4-bit quantization
# =====================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import numpy as np
from pathlib import Path
import json
import os

print("🔄 Loading MedAlpaca (optimized 4-bit mode)...")

🔄 Loading MedAlpaca (optimized 4-bit mode)...


In [3]:
# MedAlpaca configuration
model_name = "medalpaca/medalpaca-7b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Create generation pipeline
medalpaca_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    device_map="auto"
)

print("✅ MedAlpaca loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Device set to use cuda:0


✅ MedAlpaca loaded successfully!


In [4]:
# =====================================
# STEP 3: Load RAG System
# =====================================
print("🔄 Loading Medical RAG System...")

# Load FAISS index and embedding model
FAISS_PICKLE_PATH = "/content/drive/MyDrive/medical_faiss.pkl"

if os.path.exists(FAISS_PICKLE_PATH):
    with open(FAISS_PICKLE_PATH, "rb") as f:
        rag_data = pickle.load(f)

    rag_index = rag_data["index"]
    rag_questions = rag_data["questions"]
    rag_answers = rag_data["answers"]
    rag_sources = rag_data["sources"]

    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    print(f"✅ RAG system loaded with {len(rag_questions)} medical entries")
else:
    print("⚠️ RAG index not found. Running in MedAlpaca-only mode.")
    rag_index = None

🔄 Loading Medical RAG System...
⚠️ RAG index not found. Running in MedAlpaca-only mode.


In [5]:
# =====================================
# STEP 4: Enhanced Retrieval Function
# =====================================
def retrieve_medical_context(user_question, k=5, threshold=0.65):
    """Enhanced retrieval with better context selection"""
    if rag_index is None:
        return []

    # Encode query
    q_emb = embed_model.encode([user_question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    # Search with more candidates initially
    D, I = rag_index.search(q_emb, k*2)
    sims = D[0].tolist()
    idxs = I[0].tolist()

    candidates = []
    for s, idx in zip(sims, idxs):
        if idx < 0 or idx >= len(rag_questions) or s < threshold:
            continue
        candidates.append({
            "question": rag_questions[idx],
            "answer": rag_answers[idx],
            "source": rag_sources[idx],
            "score": float(s)
        })

    # Return top k relevant candidates
    return sorted(candidates, key=lambda x: x["score"], reverse=True)[:k]

In [6]:
# =====================================
# STEP 5: Smart Integration Engine
# =====================================
def smart_medical_response(user_input):
    """
    Intelligent integration of both models:
    1. Use RAG for factual, specific medical information
    2. Use MedAlpaca for complex reasoning, explanations, and follow-ups
    3. Combine strengths for comprehensive answers
    """

    # Medical disclaimer (always included)
    disclaimer = "\n\n⚠️ **Disclaimer**: I am an AI assistant. Always consult healthcare professionals for medical advice."

    # Step 1: Retrieve relevant medical context
    retrieved_context = retrieve_medical_context(user_input, k=3, threshold=0.6)

    # Step 2: Analyze query type
    query_lower = user_input.lower()
    is_factual = any(keyword in query_lower for keyword in [
        'what is', 'symptoms of', 'treatment for', 'causes of',
        'definition of', 'diagnosis of', 'medication for'
    ])

    is_complex = any(keyword in query_lower for keyword in [
        'explain', 'why', 'how does', 'compare', 'difference between',
        'should i', 'can i', 'what would happen if'
    ])

    # Step 3: Generate appropriate response based on context and query type
    if retrieved_context and is_factual:
        # Use RAG for factual answers enhanced by MedAlpaca
        context_text = "\n".join([
            f"Context {i+1}: {ctx['answer']}"
            for i, ctx in enumerate(retrieved_context[:2])
        ])

        enhanced_prompt = f"""Based on the following medical information:

{context_text}

User Question: {user_input}

Please provide a clear, accurate answer summarizing the key information:"""

        response = medalpaca_pipe(enhanced_prompt)[0]["generated_text"]
        final_answer = response.split(":")[-1].strip()

        # Add source references
        if len(retrieved_context) > 0:
            sources = ", ".join(set([ctx.get('source', 'Medical Database')
                                   for ctx in retrieved_context[:2]]))
            final_answer += f"\n\n📚 Source: {sources}"

    elif retrieved_context and is_complex:
        # Use both systems for complex reasoning
        context_text = "\n".join([
            f"- {ctx['answer']}" for ctx in retrieved_context[:3]
        ])

        reasoning_prompt = f"""As a medical expert, analyze this information:

{context_text}

Now answer this complex question: "{user_input}"

Provide a detailed explanation that connects the concepts:"""

        response = medalpaca_pipe(reasoning_prompt)[0]["generated_text"]
        final_answer = response.split(":")[-1].strip()

    else:
        # Use MedAlpaca's general medical knowledge
        system_prompt = """You are MedBot Expert, an advanced medical AI assistant.
Provide helpful, accurate medical information while clearly stating limitations.
Be concise yet informative, and always prioritize patient safety."""

        full_prompt = f"{system_prompt}\n\nUser: {user_input}\nAssistant:"
        response = medalpaca_pipe(full_prompt)[0]["generated_text"]
        final_answer = response.split("Assistant:")[-1].strip()

    # Step 4: Post-process and ensure quality
    final_answer = final_answer.split('\n\n')[0]  # Take first coherent paragraph
    if len(final_answer.split()) < 10:  # If too short, use fallback
        fallback_prompt = f"Please provide a comprehensive answer to: {user_input}"
        response = medalpaca_pipe(fallback_prompt)[0]["generated_text"]
        final_answer = response.split(":")[-1].strip()

    return final_answer + disclaimer

In [7]:
# =====================================
# STEP 6: Advanced Chat Interface
# =====================================
def medical_chatbot():
    print("\n" + "="*50)
    print("🩺 SUPER MEDBOT: Integrated Medical Assistant")
    print("="*50)
    print("I combine deep medical knowledge with factual data!")
    print("Type 'quit' to exit, 'sources' for references\n")

    conversation_history = []

    while True:
        try:
            user_input = input("\nYou: ").strip()

            if user_input.lower() in ['quit', 'exit', 'stop']:
                print("\nMedBot: Thank you for chatting! Remember to consult healthcare professionals for personal medical concerns.")
                break

            elif user_input.lower() == 'sources':
                print("\nMedBot: I use two advanced systems:")
                print("• MedAlpaca 7B: Specialized medical language model")
                print("• Medical RAG: Database of verified medical information")
                continue

            elif not user_input:
                continue

            # Show typing indicator
            print("MedBot: 🤔 Analyzing...", end='', flush=True)

            # Generate response
            response = smart_medical_response(user_input)

            # Clear typing indicator and show response
            print('\r' + ' '*20 + '\r', end='')  # Clear line
            print("MedBot:", response)

            # Maintain conversation context
            ##conversation_history.append(f"User: {user_input}")
            #conversation_history.append(f"Assistant: {response}")

        except KeyboardInterrupt:
            print("\n\nMedBot: Session ended. Stay healthy!")
            break
        except Exception as e:
            print(f"\nMedBot: I encountered an error. Please try again. ({str(e)})")

In [8]:
# =====================================
# STEP 7: Quick Test Function
# =====================================
def test_integration():
    """Test the integrated system with sample questions"""
    test_questions = [
        "What are the symptoms of diabetes?",
        "Explain how antibiotics work",
        "What's the difference between COVID and flu?",
        "When should someone go to ER for chest pain?"
    ]

    print("\n🧪 Testing Integrated System...")
    for i, question in enumerate(test_questions, 1):
        print(f"\nTest {i}: {question}")
        response = smart_medical_response(question)
        print(f"Answer: {response[:200]}...")

    print("\n✅ Integration test completed!")

In [ ]:
# =====================================
# STEP 8: Launch the Super Chatbot
# =====================================
if __name__ == "__main__":
    # Run quick test
    test_integration()

    # Start interactive chat
    medical_chatbot()


🧪 Testing Integrated System...

Test 1: What are the symptoms of diabetes?
Answer: Diabetes is a metabolic disease that affects the way your body processes food for energy. It is characterized by high levels of glucose (sugar) in the blood. Symptoms of diabetes may include increased...

Test 2: Explain how antibiotics work
Answer: Antibiotics work by targeting the bacterial cell wall, which is a crucial part of the bacterial cell membrane. This disrupts the cell wall and ultimately leads to bacterial cell death.

⚠️ **Disclaime...

Test 3: What's the difference between COVID and flu?
Answer: Both are respiratory illnesses, but COVID is caused by a virus whereas flu is caused by bacteria. The symptoms of COVID include fever, cough, and shortness of breath. The symptoms of flu include fever...

Test 4: When should someone go to ER for chest pain?
Answer: Chest pain is a symptom that could indicate a number of serious medical conditions, including heart attack, pulmonary embolism, and mu

In [ ]:
medical_chatbot()
